In [1]:
"""
LongEval 2025 - Month 1 with All Rerankers
BM25 + XGBoost + Naive Bayes + RankNet + ListNet + CrossEncoder
"""

import pyterrier as pt
import pandas as pd
import os
import numpy as np
import re
import glob
import xgboost as xgb
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import torch
import torch.nn as nn
from sentence_transformers import CrossEncoder

print("=" * 80)
print("LONGEVAL 2025 - MONTH 1 WITH ALL RERANKERS")
print("=" * 80)

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if not pt.java.started():
    pt.java.init()

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_BASE = "/home/cs_lobiu001/Longeval_2025_Train_Collection_p1/release_2025_p1/French/LongEval Train Collection"
TRAIN_MONTH = "2022-06"
TRAIN_QUERIES = f"{TRAIN_BASE}/queries/{TRAIN_MONTH}_queries.txt"
TRAIN_QRELS = f"{TRAIN_BASE}/qrels/{TRAIN_MONTH}_fr/qrels_processed.txt"

TEST_BASE = "/home/cs_lobiu001/LongEval_Web_2025_Test_Collection/LongEval Test Collection"
TEST_MONTH = "2023-03"
TEST_QUERIES = f"{TEST_BASE}/queries/{TEST_MONTH}_queries.txt"

TRAIN_INDEX = "./train_index"
TEST_INDEX = "./test_index"

print(f"Configuration:")
print(f"  Train: {TRAIN_MONTH}")
print(f"  Test: {TEST_MONTH}")

# ============================================================================
# QUERY PREPROCESSING
# ============================================================================

def preprocess_query(query_text):
    query_text = query_text.replace("'", " ").replace("'", " ").replace('"', " ")
    query_text = re.sub(r'[^\w\sàâäæçéèêëïîôùûüÿœ-]', ' ', query_text)
    query_text = ' '.join(query_text.split())
    return query_text

def load_queries(query_file):
    queries = []
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t', 1) if '\t' in line else line.split(' ', 1)
            if len(parts) == 2:
                qid = parts[0].strip()
                query_text = preprocess_query(parts[1].strip())
                queries.append({'qid': qid, 'query': query_text})
    return pd.DataFrame(queries)

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING DATA")
print("=" * 80)

train_queries = load_queries(TRAIN_QUERIES)
test_queries = load_queries(TEST_QUERIES)
print(f"Train queries: {len(train_queries)}")
print(f"Test queries: {len(test_queries)}")

# Load qrels
qrels_file = TRAIN_QRELS
if not os.path.exists(qrels_file):
    qrels_file = qrels_file.replace('qrels_processed.txt', 'qrels.txt')

qrels = pd.read_csv(
    qrels_file, sep=r'\s+',
    names=['qid', 'iter', 'docno', 'label'],
    dtype={'qid': str, 'docno': str, 'label': int}
)[['qid', 'docno', 'label']]
print(f"Qrels: {len(qrels)} judgments")

# ============================================================================
# LOAD INDEXES
# ============================================================================

print("\n" + "=" * 80)
print("LOADING INDEXES")
print("=" * 80)

train_index = pt.IndexFactory.of(TRAIN_INDEX)
test_index = pt.IndexFactory.of(TEST_INDEX)

print(f"✓ Train index: {train_index.getCollectionStatistics().getNumberOfDocuments()} docs")
print(f"✓ Test index: {test_index.getCollectionStatistics().getNumberOfDocuments()} docs")

# Fix docno prefix if needed
meta = train_index.getMetaIndex()
sample_index_docs = [meta.getItem("docno", i) for i in range(min(10, meta.size()))]
sample_qrel_docs = qrels['docno'].head(10).tolist()

print(f"\nSample index docnos: {sample_index_docs[:3]}")
print(f"Sample qrel docnos: {sample_qrel_docs[:3]}")

index_has_prefix = any(str(d).startswith('doc') for d in sample_index_docs if d)
qrels_has_prefix = any(str(d).startswith('doc') for d in sample_qrel_docs if d)

if index_has_prefix and not qrels_has_prefix:
    print("⚠️  Adding 'doc' prefix to qrels...")
    qrels['docno'] = 'doc' + qrels['docno'].astype(str)
elif not index_has_prefix and qrels_has_prefix:
    print("⚠️  Removing 'doc' prefix from qrels...")
    qrels['docno'] = qrels['docno'].str.replace('^doc', '', regex=True)

# ============================================================================
# BM25 RETRIEVAL
# ============================================================================

print("\n" + "=" * 80)
print("BM25 RETRIEVAL")
print("=" * 80)

train_bm25 = pt.terrier.Retriever(train_index, wmodel="BM25", num_results=100)
test_bm25 = pt.terrier.Retriever(test_index, wmodel="BM25", num_results=100)

print("Running BM25 on train queries...")
train_results = train_bm25.transform(train_queries)
print(f"✓ Train results: {len(train_results)}")

print("Running BM25 on test queries...")
test_results = test_bm25.transform(test_queries)
print(f"✓ Test results: {len(test_results)}")

# ============================================================================
# FEATURE CREATION
# ============================================================================

print("\n" + "=" * 80)
print("CREATING FEATURES")
print("=" * 80)

query_lens = train_queries.set_index('qid')['query'].str.split().str.len()
feature_cols = ['score', 'query_len', 'rank_score', 'score_norm', 'score_log']

def add_features(results, query_lens_map):
    results = results.copy()
    results['query_len'] = results['qid'].map(query_lens_map).fillna(3)
    results['rank_score'] = 1.0 / (results['rank'] + 1)
    results['score_norm'] = results['score'] / results.groupby('qid')['score'].transform('max')
    results['score_log'] = np.log1p(np.clip(results['score'], 0, None))
    return results

train_features = add_features(train_results, query_lens)
train_data = train_features.merge(qrels, on=['qid', 'docno'], how='left')
train_data['label'] = train_data['label'].fillna(0).astype(int)

X_train = train_data[feature_cols].fillna(0)
y_train = train_data['label']
train_groups = train_data.groupby('qid').size().values

print(f"Training samples: {len(X_train)}")
print(f"Positive labels: {(y_train > 0).sum()}")

# Scale features for neural models
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ============================================================================
# 1. XGBOOST RERANKER
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING XGBOOST")
print("=" * 80)

xgb_ranker = xgb.XGBRanker(
    objective='rank:ndcg',
    learning_rate=0.1,
    n_estimators=50,
    max_depth=6,
    random_state=42,
    verbosity=0
)
xgb_ranker.fit(X_train, y_train, group=train_groups)
print("✓ XGBoost trained!")

# ============================================================================
# 2. NAIVE BAYES RERANKER
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING NAIVE BAYES")
print("=" * 80)

y_train_binary = (y_train > 0).astype(int)

nb_classifier = GaussianNB()
nb_classifier.fit(X_train_scaled, y_train_binary)
print("✓ Naive Bayes trained!")

# ============================================================================
# 3. RANKNET (Neural Pairwise Ranking)
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING RANKNET")
print("=" * 80)

class RankNetModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)

ranknet_model = RankNetModel(len(feature_cols)).to(device)
optimizer = torch.optim.Adam(ranknet_model.parameters(), lr=0.001)

X_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_tensor = torch.FloatTensor(y_train.values).to(device)

ranknet_model.train()
for epoch in range(10):
    optimizer.zero_grad()
    scores = ranknet_model(X_tensor).squeeze()
    loss = nn.MSELoss()(scores, y_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        print(f"  Epoch {epoch}: Loss = {loss.item():.4f}")

ranknet_model.eval()
print("✓ RankNet trained!")

# ============================================================================
# 4. LISTNET (Listwise Ranking)
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING LISTNET")
print("=" * 80)

class ListNetModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)

listnet_model = ListNetModel(len(feature_cols)).to(device)
optimizer = torch.optim.Adam(listnet_model.parameters(), lr=0.001)

listnet_model.train()
for epoch in range(10):
    total_loss = 0
    qids = train_data['qid'].unique()
    
    for qid in qids[:1000]:  # Limit for speed
        mask = train_data['qid'] == qid
        if mask.sum() < 2:
            continue
        
        X_q = torch.FloatTensor(X_train_scaled[mask]).to(device)
        y_q = torch.FloatTensor(train_data.loc[mask, 'label'].values).to(device)
        
        optimizer.zero_grad()
        scores = listnet_model(X_q).squeeze()
        
        # ListNet loss: KL divergence on softmax distributions
        pred_probs = torch.softmax(scores, dim=0)
        true_probs = torch.softmax(y_q, dim=0)
        
        loss = -torch.sum(true_probs * torch.log(pred_probs + 1e-10))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        print(f"  Epoch {epoch}: Loss = {total_loss/1000:.4f}")

listnet_model.eval()
print("✓ ListNet trained!")

# ============================================================================
# 5. CROSS-ENCODER (Multilingual)
# ============================================================================

print("\n" + "=" * 80)
print("LOADING CROSS-ENCODER")
print("=" * 80)

cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=device)
print("✓ CrossEncoder loaded (multilingual mMiniLM)!")

# Load document texts for CrossEncoder (on-the-fly from TREC files)
print("\nLoading document texts for CrossEncoder...")

TRAIN_DOCS = f"{TRAIN_BASE}/Trec/{TRAIN_MONTH}_fr"
TEST_DOCS_JSON = f"{TEST_BASE}/Json/{TEST_MONTH}_fr/collection"

def load_doc_texts_trec(trec_dir, max_files=None):
    """Load document texts from TREC files into a dictionary."""
    doc_texts = {}
    files = sorted(glob.glob(os.path.join(trec_dir, "*.trec")))
    if not files:
        # Try subdirectory
        files = sorted(glob.glob(os.path.join(trec_dir, "collection", "*.trec")))
    if max_files:
        files = files[:max_files]
    
    doc_pattern = re.compile(r'<DOC>(.*?)</DOC>', re.DOTALL)
    docno_pattern = re.compile(r'<DOCNO>(.*?)</DOCNO>', re.DOTALL)
    text_pattern = re.compile(r'<TEXT>(.*?)</TEXT>', re.DOTALL)
    
    for filepath in tqdm(files, desc="Loading TREC docs"):
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        for doc_match in doc_pattern.finditer(content):
            doc_content = doc_match.group(1)
            docno_match = docno_pattern.search(doc_content)
            text_match = text_pattern.search(doc_content)
            
            if docno_match and text_match:
                docno = docno_match.group(1).strip()
                text = text_match.group(1).strip()[:1000]  # Truncate for memory
                if docno and text:
                    doc_texts[docno] = text
    
    print(f"Loaded {len(doc_texts)} document texts")
    return doc_texts

def load_doc_texts_json(json_dir, max_files=None):
    """Load document texts from JSON files into a dictionary."""
    import json
    doc_texts = {}
    files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
    if max_files:
        files = files[:max_files]
    
    for filepath in tqdm(files, desc="Loading JSON docs"):
        with open(filepath, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        docno = str(item.get('id') or item.get('docno') or item.get('_id'))
                        text = item.get('contents') or item.get('text') or item.get('body', '')
                        if docno and text:
                            doc_texts[docno] = text[:1000]
                else:
                    docno = str(data.get('id') or data.get('docno'))
                    text = data.get('contents') or data.get('text', '')
                    if docno and text:
                        doc_texts[docno] = text[:1000]
            except:
                pass
    
    print(f"Loaded {len(doc_texts)} document texts")
    return doc_texts

# Load texts (limit files for speed - adjust as needed)
train_doc_texts = load_doc_texts_trec(TRAIN_DOCS, max_files=50)
test_doc_texts = load_doc_texts_json(TEST_DOCS_JSON, max_files=25)

# ============================================================================
# RERANKER TRANSFORMERS
# ============================================================================

print("\n" + "=" * 80)
print("CREATING RERANKER TRANSFORMERS")
print("=" * 80)

all_query_lens = {
    **train_queries.set_index('qid')['query'].str.split().str.len().to_dict(),
    **test_queries.set_index('qid')['query'].str.split().str.len().to_dict()
}

class XGBoostReranker(pt.Transformer):
    def __init__(self, model, feature_cols, query_lens_map):
        self.model = model
        self.feature_cols = feature_cols
        self.query_lens_map = query_lens_map
    
    def transform(self, results):
        if len(results) == 0:
            return results
        results = add_features(results, self.query_lens_map)
        X = results[self.feature_cols].fillna(0)
        results['score'] = self.model.predict(X)
        results = results.sort_values(['qid', 'score'], ascending=[True, False])
        results['rank'] = results.groupby('qid').cumcount()
        return results[['qid', 'docno', 'score', 'rank']]

class NaiveBayesReranker(pt.Transformer):
    def __init__(self, model, scaler, feature_cols, query_lens_map):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.query_lens_map = query_lens_map
    
    def transform(self, results):
        if len(results) == 0:
            return results
        results = add_features(results, self.query_lens_map)
        X = self.scaler.transform(results[self.feature_cols].fillna(0))
        results['score'] = self.model.predict_proba(X)[:, 1]
        results = results.sort_values(['qid', 'score'], ascending=[True, False])
        results['rank'] = results.groupby('qid').cumcount()
        return results[['qid', 'docno', 'score', 'rank']]

class RankNetReranker(pt.Transformer):
    def __init__(self, model, scaler, feature_cols, query_lens_map):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.query_lens_map = query_lens_map
    
    def transform(self, results):
        if len(results) == 0:
            return results
        results = add_features(results, self.query_lens_map)
        X = self.scaler.transform(results[self.feature_cols].fillna(0))
        X_tensor = torch.FloatTensor(X).to(device)
        with torch.no_grad():
            results['score'] = self.model(X_tensor).cpu().numpy().squeeze()
        results = results.sort_values(['qid', 'score'], ascending=[True, False])
        results['rank'] = results.groupby('qid').cumcount()
        return results[['qid', 'docno', 'score', 'rank']]

class ListNetReranker(pt.Transformer):
    def __init__(self, model, scaler, feature_cols, query_lens_map):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.query_lens_map = query_lens_map
    
    def transform(self, results):
        if len(results) == 0:
            return results
        results = add_features(results, self.query_lens_map)
        X = self.scaler.transform(results[self.feature_cols].fillna(0))
        X_tensor = torch.FloatTensor(X).to(device)
        with torch.no_grad():
            results['score'] = self.model(X_tensor).cpu().numpy().squeeze()
        results = results.sort_values(['qid', 'score'], ascending=[True, False])
        results['rank'] = results.groupby('qid').cumcount()
        return results[['qid', 'docno', 'score', 'rank']]

class CrossEncoderReranker(pt.Transformer):
    def __init__(self, model, doc_texts, query_df, batch_size=64):
        self.model = model
        self.doc_texts = doc_texts
        self.query_map = query_df.set_index('qid')['query'].to_dict()
        self.batch_size = batch_size
    
    def transform(self, results):
        if len(results) == 0:
            return results
        results = results.copy()
        
        # Get query text
        results['query_text'] = results['qid'].map(self.query_map)
        
        # Get document text
        results['doc_text'] = results['docno'].map(self.doc_texts).fillna('')
        
        # Filter out rows without text
        has_text = results['doc_text'] != ''
        if has_text.sum() == 0:
            print("⚠️ No document texts found for CrossEncoder")
            return results[['qid', 'docno', 'score', 'rank']]
        
        # Create pairs and score
        pairs = list(zip(
            results.loc[has_text, 'query_text'].tolist(),
            results.loc[has_text, 'doc_text'].str[:512].tolist()
        ))
        
        scores = self.model.predict(pairs, batch_size=self.batch_size, show_progress_bar=True)
        results.loc[has_text, 'score'] = scores
        
        # For docs without text, keep original score (or set to -inf)
        results.loc[~has_text, 'score'] = results.loc[~has_text, 'score'].fillna(-999)
        
        results = results.sort_values(['qid', 'score'], ascending=[True, False])
        results['rank'] = results.groupby('qid').cumcount()
        return results[['qid', 'docno', 'score', 'rank']]

# Create all rerankers
xgb_reranker = XGBoostReranker(xgb_ranker, feature_cols, all_query_lens)
nb_reranker = NaiveBayesReranker(nb_classifier, scaler, feature_cols, all_query_lens)
ranknet_reranker = RankNetReranker(ranknet_model, scaler, feature_cols, all_query_lens)
listnet_reranker = ListNetReranker(listnet_model, scaler, feature_cols, all_query_lens)
crossencoder_reranker_train = CrossEncoderReranker(cross_encoder, train_doc_texts, train_queries, batch_size=64)
crossencoder_reranker_test = CrossEncoderReranker(cross_encoder, test_doc_texts, test_queries, batch_size=64)

print("✓ All rerankers created!")

# ============================================================================
# EVALUATE ALL MODELS
# ============================================================================

print("\n" + "=" * 80)
print("EVALUATING ALL MODELS")
print("=" * 80)

from pyterrier.measures import P, nDCG, AP, RR

# Rerank train results
print("Reranking with XGBoost...")
train_xgb = xgb_reranker.transform(train_results)

print("Reranking with Naive Bayes...")
train_nb = nb_reranker.transform(train_results)

print("Reranking with RankNet...")
train_ranknet = ranknet_reranker.transform(train_results)

print("Reranking with ListNet...")
train_listnet = listnet_reranker.transform(train_results)

print("Reranking with CrossEncoder (this takes a while)...")
train_crossencoder = crossencoder_reranker_train.transform(train_results)

# Evaluate all
all_train_results = {
    "BM25": train_results,
    "XGBoost": train_xgb,
    "NaiveBayes": train_nb,
    "RankNet": train_ranknet,
    "ListNet": train_listnet,
    "CrossEncoder": train_crossencoder
}

print("\n📊 EVALUATION RESULTS:")
metrics_list = []
for name, res in all_train_results.items():
    res_df = pt.Transformer.from_df(res)
    metrics = pt.Experiment(
        [res_df],
        train_queries,
        qrels,
        eval_metrics=[P@10, nDCG@10, AP, RR],
        names=[name],
        filter_by_qrels=True
    )
    metrics_list.append(metrics)
    print(f"\n{name}:")
    print(metrics)

all_metrics = pd.concat(metrics_list, ignore_index=True)

# ============================================================================
# APPLY TO TEST
# ============================================================================

print("\n" + "=" * 80)
print("APPLYING TO TEST DATA")
print("=" * 80)

print("Reranking test with XGBoost...")
test_xgb = xgb_reranker.transform(test_results)

print("Reranking test with Naive Bayes...")
test_nb = nb_reranker.transform(test_results)

print("Reranking test with RankNet...")
test_ranknet = ranknet_reranker.transform(test_results)

print("Reranking test with ListNet...")
test_listnet = listnet_reranker.transform(test_results)

print("Reranking test with CrossEncoder (this takes a while)...")
test_crossencoder = crossencoder_reranker_test.transform(test_results)

# ============================================================================
# DISPLAY SAMPLE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAMPLE RESULTS (First Query)")
print("=" * 80)

sample_qid = test_results['qid'].iloc[0]
print(f"\nQuery ID: {sample_qid}")

print("\n📊 BM25:")
print(test_results[test_results['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

print("\n📊 XGBoost:")
print(test_xgb[test_xgb['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

print("\n📊 Naive Bayes:")
print(test_nb[test_nb['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

print("\n📊 RankNet:")
print(test_ranknet[test_ranknet['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

print("\n📊 ListNet:")
print(test_listnet[test_listnet['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

print("\n📊 CrossEncoder:")
print(test_crossencoder[test_crossencoder['qid'] == sample_qid][['qid', 'docno', 'score', 'rank']].head(10))

# ============================================================================
# SAVE ALL RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

submissions = {
    "bm25": test_results,
    "xgboost": test_xgb,
    "naivebayes": test_nb,
    "ranknet": test_ranknet,
    "listnet": test_listnet,
    "crossencoder": test_crossencoder
}

for name, res in submissions.items():
    filename = f"test_{name}_{TEST_MONTH}.res"
    pt.io.write_results(res, filename, format="trec", run_name=name)
    print(f"✓ {filename}")

# Save metrics
all_metrics.to_csv(f"metrics_all_models_{TRAIN_MONTH}.csv", index=False)
print(f"✓ metrics_all_models_{TRAIN_MONTH}.csv")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("🎉 RESULTS SUMMARY - ALL MODELS")
print("=" * 80)

print("\n📊 ALL METRICS:")
print(all_metrics.to_string())

print("\n📈 IMPROVEMENTS OVER BM25:")
bm25_row = all_metrics[all_metrics['name'] == 'BM25'].iloc[0]

for model_name in ["XGBoost", "NaiveBayes", "RankNet", "ListNet", "CrossEncoder"]:
    model_row = all_metrics[all_metrics['name'] == model_name].iloc[0]
    print(f"\n{model_name}:")
    for metric in ["P@10", "nDCG@10", "AP", "RR"]:
        bm25_val = bm25_row[metric]
        model_val = model_row[metric]
        if bm25_val > 0:
            imp = ((model_val - bm25_val) / bm25_val * 100)
            print(f"  {metric}: {bm25_val:.4f} → {model_val:.4f} ({imp:+.2f}%)")

print(f"\n📤 SUBMISSION FILES:")
for name in submissions.keys():
    print(f"  - test_{name}_{TEST_MONTH}.res")

print("\n✅ ALL MODELS COMPLETE!")

LONGEVAL 2025 - MONTH 1 WITH ALL RERANKERS
Device: cuda


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


Configuration:
  Train: 2022-06
  Test: 2023-03

LOADING DATA
Train queries: 24651
Test queries: 5685
Qrels: 48862 judgments

LOADING INDEXES
✓ Train index: 592901 docs
✓ Test index: 411093 docs

Sample index docnos: ['doc1', 'doc3487649', 'doc2']
Sample qrel docnos: ['14011', '96', '12433']
⚠️  Adding 'doc' prefix to qrels...

BM25 RETRIEVAL
Running BM25 on train queries...
✓ Train results: 2361815
Running BM25 on test queries...
✓ Test results: 548051

CREATING FEATURES
Training samples: 2361815
Positive labels: 8655

TRAINING XGBOOST
✓ XGBoost trained!

TRAINING NAIVE BAYES
✓ Naive Bayes trained!

TRAINING RANKNET
  Epoch 0: Loss = 0.0204
  Epoch 5: Loss = 0.0115
✓ RankNet trained!

TRAINING LISTNET
  Epoch 0: Loss = 4.5295
  Epoch 5: Loss = 4.5292
✓ ListNet trained!

LOADING CROSS-ENCODER
✓ CrossEncoder loaded (multilingual mMiniLM)!

Loading document texts for CrossEncoder...


Loading TREC docs: 100%|████████████████████████| 50/50 [00:49<00:00,  1.02it/s]


Loaded 585882 document texts


Loading JSON docs: 100%|████████████████████████| 25/25 [00:13<00:00,  1.90it/s]


Loaded 392947 document texts

CREATING RERANKER TRANSFORMERS
✓ All rerankers created!

EVALUATING ALL MODELS
Reranking with XGBoost...
Reranking with Naive Bayes...
Reranking with RankNet...
Reranking with ListNet...
Reranking with CrossEncoder (this takes a while)...


Batches:   0%|          | 0/36904 [00:00<?, ?it/s]


📊 EVALUATION RESULTS:

BM25:
   name      P@10   nDCG@10        AP        RR
0  BM25  0.045192  0.158804  0.128893  0.184738

XGBoost:
      name      P@10   nDCG@10        AP        RR
0  XGBoost  0.046463  0.166805  0.135758  0.195393

NaiveBayes:
         name      P@10   nDCG@10        AP        RR
0  NaiveBayes  0.045192  0.158688  0.128722  0.184396

RankNet:
      name     P@10   nDCG@10        AP        RR
0  RankNet  0.00405  0.015742  0.017844  0.028019

ListNet:
      name      P@10  nDCG@10        AP        RR
0  ListNet  0.045146  0.15872  0.128881  0.184792

CrossEncoder:
           name      P@10   nDCG@10      AP        RR
0  CrossEncoder  0.052006  0.204729  0.1722  0.244054

APPLYING TO TEST DATA
Reranking test with XGBoost...
Reranking test with Naive Bayes...
Reranking test with RankNet...
Reranking test with ListNet...
Reranking test with CrossEncoder (this takes a while)...
⚠️ No document texts found for CrossEncoder

SAMPLE RESULTS (First Query)

Query ID: 3

📊 